# Training auf Google Colab

In [ ]:
# 1) GPU pruefen 
!nvidia-smi

In [ ]:
# 2) Google Drive 
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Repo klonen
REPO = "https://github.com/wieland-github/information_extraction_german_medical_data.git"
!rm -rf /content/information_extraction_german_medical_data
!git clone {REPO} /content/information_extraction_german_medical_data
%cd /content/information_extraction_german_medical_data

In [ ]:
# 4) Abhaengigkeiten installieren (GPU-Variante: cupy-cuda12x + transformers)
!pip install -q "spacy[cuda12x,transformers]>=3.8.14,<3.9.0" "datasets>=2.16.0,<3.0.0"

In [ ]:
# 5) Daten von Hugging Face laden und in spaCy-Format konvertieren
!python related_work/scripts/data_preparation.py --outdir related_work/data
!ls -lh related_work/data

In [ ]:
# 6) Training
MODEL = "deepset/gbert-large"   # oder: "deepset/gbert-base"
TEST  = False                   # True = 50 steps, False = volles Training

name  = MODEL.split("/")[-1] + ("-test" if TEST else "")
OUT   = f"./models/{name}"
extra = "--training.max_steps 50 --training.eval_frequency 25 --training.patience 0" if TEST else ""

%cd /content/information_extraction_german_medical_data/related_work
cmd = (
    f"python -m spacy train config_spacy.cfg "
    f"--components.transformer.model.name {MODEL} "
    f"--output {OUT} "
    f"--paths.train ./data/train.spacy "
    f"--paths.dev ./data/validation.spacy "
    f"--gpu-id 0 "
    f"--training.optimizer.learn_rate.initial_rate 5e-5 "
    f"--training.batcher.size 128 {extra}"
)
print(cmd)
!{cmd}

In [ ]:
# 7) Modell als ZIP in Google Drive speichern
import shutil, os

src     = f"/content/information_extraction_german_medical_data/related_work/models/{name}/model-best"
dst_dir = "/content/drive/MyDrive/gbert_colab"
os.makedirs(dst_dir, exist_ok=True)
dst = f"{dst_dir}/{name}-model-best"

shutil.make_archive(dst, "zip", src)
print("Gespeichert in Drive:", dst + ".zip")

## Lokal weiterverwenden

1. ZIP aus Google Drive herunterladen: `MyDrive/gbert_colab/<name>-model-best.zip`.
2. Im Projekt entpacken nach `related_work/models/<name>/model-best/`
   (Ordner ggf. anlegen — der **Inhalt** des ZIP entspricht dem Inhalt von `model-best/`).
3. Lokal testen (CPU):
   ```bash
   pip install -r requirements.txt
   bash related_work/scripts/evaluate_gbert_large.sh -1
   ```
   Hinweis: In `evaluate_gbert_large.sh` zeigt `MODEL_DIR` aktuell auf `./models/gbert-large-test/model-best`.
   Fuer das volle Modell dort auf `./models/gbert-large/model-best` aendern.